# Chapter 7: Fine-tuning to Follow Instructions

Companion notebook for [Chapter 7 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/instructions). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## Setup

Code carried over from earlier chapters, plus anything this chapter needs to run. Run it once and move on.

In [ ]:
#@title Setup (run me first)

import math
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- from Chapter 2 ----
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class NextTokenDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.inputs, self.targets = [], []
        for start in range(0, len(ids) - context_length, stride):
            window = ids[start : start + context_length + 1]
            self.inputs.append(torch.tensor(window[:-1]))   # all but the last
            self.targets.append(torch.tensor(window[1:]))   # all but the first

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def create_dataloader(text, batch_size=8, max_length=256, stride=256,
                      shuffle=True, drop_last=True):
    dataset = NextTokenDataset(text, tiktoken.get_encoding("gpt2"),
                               context_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last)

# ---- from Chapter 3 ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def split_heads(self, t):
        b, n, _ = t.shape
        # (b, n, d_out) -> (b, n, heads, head_dim) -> (b, heads, n, head_dim)
        return t.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.split_heads(self.to_q(x))
        k = self.split_heads(self.to_k(x))
        v = self.split_heads(self.to_v(x))

        scores = q @ k.transpose(2, 3)                    # (b, heads, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / self.head_dim ** 0.5, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                 # (b, heads, n, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)  # concat heads
        return self.out_proj(out)                         # (b, n, d_out)

# ---- from Chapter 4 ----
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # number of tokens the BPE tokenizer knows
    "context_length": 1024,  # longest input the model can look at
    "emb_dim": 768,          # length of every token vector
    "n_heads": 12,           # attention heads per block
    "n_layers": 12,          # how many transformer blocks to stack
    "drop_rate": 0.1,        # dropout probability
    "qkv_bias": True,        # bias terms in the query/key/value projections
}

class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))   # γ
        self.shift = nn.Parameter(torch.zeros(dim))  # β

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

import math

class GELU(nn.Module):
    def forward(self, x):
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x ** 3)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),  # expand:  768 → 3072
            GELU(),
            nn.Linear(4 * d, d),  # project: 3072 → 768
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.norm1 = LayerNorm(d)
        self.attn = MultiHeadAttention(
            d_in=d,
            d_out=d,
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.norm2 = LayerNorm(d)
        self.ff = FeedForward(cfg)
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # First half: tokens exchange information (attention) + shortcut
        x = x + self.drop(self.attn(self.norm1(x)))
        # Second half: each token is processed on its own (FFN) + shortcut
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        n_tokens = token_ids.shape[1]
        positions = torch.arange(n_tokens, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(positions)  # [batch, n_tokens, 768]
        x = self.drop(x)
        x = self.blocks(x)                                     # 12 blocks, shape unchanged
        x = self.final_norm(x)
        return self.out_head(x)                                # [batch, n_tokens, 50257]

def generate_greedy(model, token_ids, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        window = token_ids[:, -context_size:]      # 1. crop
        with torch.no_grad():
            logits = model(window)                 # 2. [batch, n_tokens, vocab]
        last = logits[:, -1, :]                    # 3. next-token scores
        probs = torch.softmax(last, dim=-1)        # 4. probabilities...
        next_id = torch.argmax(probs, dim=-1, keepdim=True)  # ...and the winner
        token_ids = torch.cat([token_ids, next_id], dim=1)   # 5. append
    return token_ids

# ---- from Chapter 5 ----
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(ids).unsqueeze(0)     # add a batch dimension

def token_ids_to_text(token_ids, tokenizer):
    ids = token_ids.squeeze(0).tolist()       # drop the batch dimension
    return tokenizer.decode(ids)

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    n = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    total = 0.0
    for i, (x, y) in enumerate(data_loader):
        if i >= n:
            break
        total += calc_loss_batch(x, y, model, device).item()
    return total / n

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, tokens_seen = [], [], []
    seen, step = 0, -1

    for epoch in range(num_epochs):
        model.train()                              # enable dropout
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()                  # clear old gradients
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()                        # compute new gradients
            optimizer.step()                       # update the weights
            seen += input_batch.numel()
            step += 1

            if step % eval_freq == 0:
                tr, va = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(tr)
                val_losses.append(va)
                tokens_seen.append(seen)
                print(f"epoch {epoch+1} step {step:05d}: train {tr:.3f}, val {va:.3f}")

        # after every epoch, look at what the model writes
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    idx = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        out = generate_greedy(model, idx, max_new_tokens=40, context_size=context_size)
    print(token_ids_to_text(out, tokenizer).replace("\n", " "))
    model.train()

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]          # crop to the context window
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]      # logits for the last position

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            cutoff = top_logits[:, -1:]
            logits = torch.where(logits < cutoff,
                                 torch.full_like(logits, float("-inf")), logits)

        if temperature > 0.0:
            probas = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
            break                                   # stop at end-of-text
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

import json
import os
import urllib.request
from safetensors.torch import load_file

GPT2_REPOS = {"124M": "gpt2", "355M": "gpt2-medium", "774M": "gpt2-large", "1558M": "gpt2-xl"}

def download_and_load_gpt2(model_size, models_dir="gpt2"):
    """Downloads GPT-2 weights once, then returns (settings, params) from disk."""
    folder = os.path.join(models_dir, model_size)
    os.makedirs(folder, exist_ok=True)
    for file in ("config.json", "model.safetensors"):
        path = os.path.join(folder, file)
        if not os.path.exists(path):
            url = f"https://huggingface.co/openai-community/{GPT2_REPOS[model_size]}/resolve/main/{file}"
            print("Downloading", url)
            urllib.request.urlretrieve(url, path)
    with open(os.path.join(folder, "config.json")) as f:
        settings = json.load(f)
    params = load_file(os.path.join(folder, "model.safetensors"))  # name -> tensor
    return settings, params

def assign(param, tensor):
    if param.shape != tensor.shape:
        raise ValueError(f"Shape mismatch: model {tuple(param.shape)} vs checkpoint {tuple(tensor.shape)}")
    return torch.nn.Parameter(tensor.detach().clone())

def load_weights_into_gpt(gpt, params):
    p = {k.removeprefix("transformer."): v for k, v in params.items()}
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, p["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, p["wpe.weight"])

    for i, block in enumerate(gpt.blocks):
        h = f"h.{i}."
        # OpenAI stores linear weights as (in, out); nn.Linear expects (out, in), hence .T
        # Queries, keys and values live in one combined matrix: split it in three.
        q_w, k_w, v_w = p[h + "attn.c_attn.weight"].chunk(3, dim=-1)
        q_b, k_b, v_b = p[h + "attn.c_attn.bias"].chunk(3, dim=-1)
        attn = block.attn
        attn.to_q.weight, attn.to_q.bias = assign(attn.to_q.weight, q_w.T), assign(attn.to_q.bias, q_b)
        attn.to_k.weight, attn.to_k.bias = assign(attn.to_k.weight, k_w.T), assign(attn.to_k.bias, k_b)
        attn.to_v.weight, attn.to_v.bias = assign(attn.to_v.weight, v_w.T), assign(attn.to_v.bias, v_b)
        attn.out_proj.weight = assign(attn.out_proj.weight, p[h + "attn.c_proj.weight"].T)
        attn.out_proj.bias = assign(attn.out_proj.bias, p[h + "attn.c_proj.bias"])

        expand, project = block.ff.net[0], block.ff.net[2]
        expand.weight = assign(expand.weight, p[h + "mlp.c_fc.weight"].T)
        expand.bias = assign(expand.bias, p[h + "mlp.c_fc.bias"])
        project.weight = assign(project.weight, p[h + "mlp.c_proj.weight"].T)
        project.bias = assign(project.bias, p[h + "mlp.c_proj.bias"])

        block.norm1.scale = assign(block.norm1.scale, p[h + "ln_1.weight"])
        block.norm1.shift = assign(block.norm1.shift, p[h + "ln_1.bias"])
        block.norm2.scale = assign(block.norm2.scale, p[h + "ln_2.weight"])
        block.norm2.shift = assign(block.norm2.shift, p[h + "ln_2.bias"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, p["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, p["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, p["wte.weight"])  # GPT-2 ties these

## What instruction fine-tuning does

## The dataset

In [ ]:
import json
import os
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
       "main/ch07/01_main-chapter-code/instruction-data.json")
if not os.path.exists("instruction-data.json"):
    urllib.request.urlretrieve(url, "instruction-data.json")

with open("instruction-data.json") as f:
    data = json.load(f)

print(len(data))   # 1100
print(data[42])
# {'instruction': 'What is the formula for area of a triangle?',
#  'input': '',
#  'output': 'The formula for the area of a triangle is 1/2 * base * height.'}

### A prompt template

In [ ]:
def format_prompt(entry):
    text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    if entry["input"]:
        text += f"\n\n### Input:\n{entry['input']}"
    return text

entry = data[42]
full_text = format_prompt(entry) + f"\n\n### Response:\n{entry['output']}"

In [ ]:
n_train = int(len(data) * 0.85)
n_test = int(len(data) * 0.10)

train_data = data[:n_train]
test_data = data[n_train:n_train + n_test]
val_data = data[n_train + n_test:]
print(len(train_data), len(val_data), len(test_data))   # 935 55 110

## Batching with a custom collate function

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.encoded = [
            tokenizer.encode(format_prompt(e) + f"\n\n### Response:\n{e['output']}")
            for e in data
        ]

    def __getitem__(self, idx):
        return self.encoded[idx]

    def __len__(self):
        return len(self.encoded)

### Why −100?

In [ ]:
import torch.nn.functional as F

logits = torch.tensor([[-1.0, 1.0], [-0.5, 1.5], [0.2, 0.1]])
targets = torch.tensor([0, 1, 1])
print(F.cross_entropy(logits, targets))           # averages all 3 rows

targets = torch.tensor([0, 1, -100])
print(F.cross_entropy(logits, targets))           # same as using only the first 2 rows

### The collate function

In [ ]:
def collate(batch, pad_id=50256, ignore_index=-100,
            allowed_max_length=None, device="cpu"):
    # +1 because every sequence gets one end-of-text token appended
    longest = max(len(seq) + 1 for seq in batch)
    inputs, targets = [], []

    for seq in batch:
        seq = seq + [pad_id]                           # the end-of-text token
        seq = seq + [pad_id] * (longest - len(seq))    # padding
        x = torch.tensor(seq[:-1])
        y = torch.tensor(seq[1:])

        # Keep the first end-of-text target, ignore the rest
        pad_positions = torch.nonzero(y == pad_id).squeeze(1)
        if pad_positions.numel() > 1:
            y[pad_positions[1:]] = ignore_index

        if allowed_max_length is not None:
            x, y = x[:allowed_max_length], y[:allowed_max_length]
        inputs.append(x)
        targets.append(y)

    return torch.stack(inputs).to(device), torch.stack(targets).to(device)

In [ ]:
from functools import partial

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
customized_collate = partial(collate, device=device, allowed_max_length=1024)

torch.manual_seed(123)
train_loader = DataLoader(
    InstructionDataset(train_data, tokenizer), batch_size=8,
    collate_fn=customized_collate, shuffle=True, drop_last=True,
)
val_loader = DataLoader(InstructionDataset(val_data, tokenizer), batch_size=8,
                        collate_fn=customized_collate)

## Fine-tuning a larger model

In [ ]:
BASE_CONFIG = {
    "vocab_size": 50257, "context_length": 1024,
    "drop_rate": 0.0, "qkv_bias": True,
    "emb_dim": 1024, "n_layers": 24, "n_heads": 16,   # GPT-2 medium
}

settings, params = download_and_load_gpt2(model_size="355M", models_dir="gpt2")
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.to(device)

In [ ]:
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=2, eval_freq=5, eval_iter=5,
    start_context=format_prompt(val_data[0]), tokenizer=tokenizer,
)

## Extracting and saving responses

In [ ]:
def respond(entry, model, tokenizer, max_new_tokens=256):
    prompt = format_prompt(entry) + "\n\n### Response:\n"
    ids = generate(
        model, text_to_token_ids(prompt, tokenizer).to(device),
        max_new_tokens=max_new_tokens,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,                      # stop at end-of-text
    )
    text = token_ids_to_text(ids, tokenizer)
    return text[len(prompt):].strip()

In [ ]:
from tqdm import tqdm

for entry in tqdm(test_data):
    entry["model_response"] = respond(entry, model, tokenizer)

with open("instruction-data-with-response.json", "w") as f:
    json.dump(test_data, f, indent=4)

torch.save(model.state_dict(), "gpt2-medium355M-sft.pth")

## Evaluating with an LLM judge

In [ ]:
import subprocess, time
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
subprocess.run(["ollama", "pull", "llama3"], check=True)

In [ ]:
import json
import urllib.request

def ask_ollama(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "options": {"seed": 123, "temperature": 0, "num_ctx": 2048},
        "stream": False,
    }
    request = urllib.request.Request(
        url, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read())["message"]["content"]

In [ ]:
def judge_scores(entries):
    scores = []
    for e in tqdm(entries, desc="Scoring"):
        prompt = (
            "You are grading a model's answer.\n"
            f"Instruction: {format_prompt(e)}\n"
            f"Reference answer: {e['output']}\n"
            f"Model answer: {e['model_response']}\n"
            "Score the model answer from 0 (useless) to 100 (perfect), "
            "using the reference as a guide. Reply with a single integer only."
        )
        try:
            scores.append(int(ask_ollama(prompt).strip()))
        except ValueError:
            print("Could not parse score for:", e["instruction"])
    return scores

scores = judge_scores(test_data)
print(f"Average score: {sum(scores) / len(scores):.1f}")

## Where to go from here